In [1]:
import gymnasium as gym
import numpy as np
from IPython.display import display, clear_output, Image
import time
# from PIL import Image
from collections import deque
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
# sutton_barto_reward=True gives a more correct reward func, should still be easily learnable without False
env = gym.make('CartPole-v1', render_mode='human', sutton_barto_reward=False)  

def render_frame(env):
    frame = env.render()

In [16]:
env.unwrapped.render_mode = 'human'

state, info = env.reset()
for step in range(1000):
    action = 0
    state, reward, done, truncated, info = env.step(action)
    
    clear_output(wait=True)
    print('taking action', action, ' got reward', reward)

    if done or truncated:
        print(f"Episode finished with environment reward: {reward}")
        break

taking action 0  got reward 1.0
Episode finished with environment reward: 1.0


In [17]:
print(env.action_space)
print(env.observation_space.shape)

Discrete(2)
(4,)


In [52]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)


class ReplayBuffer:
    def __init__(self, maxlen):
        self.buffer = deque(maxlen=maxlen)

    def __len__(self): 
        return len(self.buffer)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        assert batch_size <= len(self), 'sample size is greater than population of buffer'
        states, actions, rewards, next_states, dones = zip(*random.sample(self.buffer, batch_size))  # with replacement
        return torch.tensor(states), torch.tensor(actions), torch.tensor(rewards), torch.tensor(next_states), torch.tensor(dones)

In [53]:
buf = ReplayBuffer(100)
buf.push(*([5]*5))
buf.push(*([4]*5))
buf.push(*([3]*5))
buf.push(*([2]*5))
buf.sample(3)

(tensor([3, 4, 2]),
 tensor([3, 4, 2]),
 tensor([3, 4, 2]),
 tensor([3, 4, 2]),
 tensor([3, 4, 2]))